# 03 - RAG流水线 (RAG Pipeline)

本notebook介绍完整的RAG流水线实现，包括文档处理、检索和生成。

## 内容大纲

### 第一部分: 文本分块
1.1 基础分块演示
1.2 不同分隔符效果对比
1.3 分块参数调优

### 第二部分: RAG流水线
2.1 基础RAG流水线
2.2 批量文档处理
2.3 自定义配置与生成器

### 第三部分: 查询与响应
3.1 基础查询
3.2 多轮查询对比
3.3 响应结构分析

### 第四部分: 性能分析
4.1 分块效率分析
4.2 检索质量评估
4.3 上下文长度优化

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
from typing import List
from collections import Counter

from vector_store import Document
from rag_pipeline import (
    RAGConfig,
    RAGPipeline,
    RecursiveTextSplitter,
)

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

---

## 第一部分: 文本分块

### 1.1 基础分块演示

In [ ]:
# 创建递归文本分割器
splitter = RecursiveTextSplitter(chunk_size=100, chunk_overlap=20)

# 测试文本
long_text = """机器学习是人工智能的一个分支，它使计算机能够从数据中学习。

深度学习是机器学习的子领域，使用多层神经网络。

自然语言处理让计算机理解人类语言，广泛应用于翻译和问答系统。

计算机视觉让机器能够"看"和理解图像，应用于人脸识别和自动驾驶。"""

# 执行分块
chunks = splitter.split_text(long_text)

print(f"原始文本长度: {len(long_text)} 字符")
print(f"分块数量: {len(chunks)}")
print("\n" + "="*50)
for i, chunk in enumerate(chunks, 1):
    print(f"\n[分块 {i}] 长度: {len(chunk)} 字符")
    print(f"内容: {chunk}")

### 1.2 不同分隔符效果对比

In [ ]:
# 创建不同配置的分割器
configs = [
    {"name": "段落分割", "separators": ["\n\n", "\n", ""]},
    {"name": "句子分割", "separators": ["。", "！", "？", ".", "!", "?"]},
    {"name": "字符分割", "separators": [""]},
]

test_text = "这是第一段。这是第二句！这是第三句？\n\n这是第二段。继续内容。"

print("不同分隔符分块效果对比：")
print("=" * 60)

for cfg in configs:
    splitter_test = RecursiveTextSplitter(
        chunk_size=50,
        chunk_overlap=0,
        separators=cfg["separators"]
    )
    chunks_test = splitter_test.split_text(test_text)
    print(f"\n{cfg['name']} ({len(chunks_test)} 个分块):")
    for i, c in enumerate(chunks_test, 1):
        print(f"  [{i}] {c[:40]}{'...' if len(c) > 40 else ''}")

### 1.3 分块参数调优

In [ ]:
# 测试不同chunk_size和overlap的效果
sample_docs = [
    "机器学习是人工智能的核心技术，通过数据训练模型进行预测。深度学习使用神经网络。强化学习通过奖励学习策略。监督学习需要标注数据。无监督学习发现数据模式。" * 5
]

chunk_sizes = [100, 200, 300, 400]
overlaps = [0, 20, 50, 100]

results = []

for size in chunk_sizes:
    for overlap in overlaps:
        if overlap < size:
            splitter_tune = RecursiveTextSplitter(
                chunk_size=size,
                chunk_overlap=overlap
            )
            chunks_tune = splitter_tune.split_text(sample_docs[0])
            results.append({
                'chunk_size': size,
                'overlap': overlap,
                'num_chunks': len(chunks_tune),
                'avg_len': np.mean([len(c) for c in chunks_tune])
            })

# 展示结果
print("\n分块参数调优结果：")
print(f"{'chunk_size':<10} {'overlap':<10} {'分块数':<8} {'平均长度':<10}")
print("-" * 45)
for r in results:
    print(f"{r['chunk_size']:<10} {r['overlap']:<10} {r['num_chunks']:<8} {r['avg_len']:.1f}")

---

## 第二部分: RAG流水线

### 2.1 基础RAG流水线

In [ ]:
# 创建RAG配置
config = RAGConfig(
    chunk_size=200,
    chunk_overlap=20,
    top_k=3,
)

# 初始化流水线
pipeline = RAGPipeline(config=config)

# 准备文档
documents = [
    Document(content="机器学习是人工智能的核心技术，通过数据训练模型。它包括监督学习、无监督学习和强化学习三大类。"),
    Document(content="深度学习使用多层神经网络，在图像和语音识别中表现出色。常见的架构包括CNN和RNN。"),
    Document(content="自然语言处理让计算机理解人类语言，应用于翻译和问答。大语言模型如GPT推动了NLP发展。"),
    Document(content="强化学习通过奖励信号学习最优策略，用于游戏和机器人控制。Q学习是经典算法之一。"),
    Document(content="Transformer架构使用自注意力机制，是现代大语言模型的基础。它支持并行计算，训练效率高。"),
]

# 添加文档
ids = pipeline.add_documents(documents)
print(f"✓ 添加了 {len(ids)} 个文档块")
print(f"✓ 流水线配置: chunk_size={config.chunk_size}, top_k={config.top_k}")

### 2.2 批量文档处理

In [ ]:
# 使用add_texts批量添加
texts = [
    "卷积神经网络(CNN)擅长处理图像数据，通过卷积层提取特征。",
    "循环神经网络(RNN)适合序列数据，但存在梯度消失问题。",
    "LSTM和GRU是RNN的改进版本，通过门控机制解决长期依赖。",
    "注意力机制让模型关注输入的重要部分，提升了模型性能。",
    "BERT使用双向Transformer，在理解任务上表现优异。",
]

# 添加元数据
metadatas = [
    {"category": "CNN", "year": 2012},
    {"category": "RNN", "year": 1986},
    {"category": "RNN", "year": 1997},
    {"category": "Attention", "year": 2017},
    {"category": "Transformer", "year": 2018},
]

text_ids = pipeline.add_texts(texts, metadatas)
print(f"✓ 批量添加了 {len(text_ids)} 个文本块")
print(f"✓ 总文档块数: {len(ids) + len(text_ids)}")

### 2.3 自定义配置与生成器

In [ ]:
# 自定义生成器
def custom_generator(prompt: str) -> str:
    """模拟更智能的生成器"""
    # 提取上下文中的关键词
    keywords = ["机器学习", "深度学习", "强化学习", "神经网络", "Transformer"]
    found = [kw for kw in keywords if kw in prompt]
    
    if found:
        return f"根据提供的上下文，{found[0]}是一个重要的AI技术领域。它通过特定方法处理数据，在多个应用场景中表现出色。"
    return "根据上下文，这是一个关于人工智能技术的问题。"

# 创建带自定义生成器的流水线
custom_config = RAGConfig(
    chunk_size=150,
    chunk_overlap=30,
    top_k=2,
    prompt_template="""请根据以下上下文简洁回答问题。

上下文：
{context}

问题：{question}

简洁回答："""
)

custom_pipeline = RAGPipeline(
    config=custom_config,
    generator=custom_generator
)

custom_pipeline.add_documents(documents[:3])
print("✓ 自定义流水线创建成功")

---

## 第三部分: 查询与响应

### 3.1 基础查询

In [ ]:
response = pipeline.query("什么是深度学习？")

print("=" * 60)
print("📝 查询问题: 什么是深度学习？")
print("=" * 60)

print("\n🤖 回答:")
print(response.answer)

print("\n📚 来源文档:")
for i, doc in enumerate(response.source_documents, 1):
    print(f"  [{i}] {doc.content[:60]}...")
    if doc.metadata:
        print(f"      元数据: {doc.metadata}")

### 3.2 多轮查询对比

In [ ]:
# 测试多个查询
test_queries = [
    "什么是强化学习？",
    "Transformer有什么特点？",
    "CNN和RNN的区别是什么？",
    "BERT的优势在哪里？",
]

print("\n多轮查询对比：")
print("=" * 70)

for query in test_queries:
    resp = pipeline.query(query)
    print(f"\n问题: {query}")
    print(f"来源数: {len(resp.source_documents)}")
    print(f"回答: {resp.answer[:80]}..." if len(resp.answer) > 80 else f"回答: {resp.answer}")

### 3.3 响应结构分析

In [ ]:
# 详细分析响应结构
response = pipeline.query("注意力机制的作用是什么？")

print("RAGResponse 结构分析：")
print("=" * 50)
print(f"\n[1] answer (回答):")
print(f"    类型: {type(response.answer)}")
print(f"    长度: {len(response.answer)} 字符")
print(f"    内容: {response.answer}")

print(f"\n[2] source_documents (来源文档):")
print(f"    类型: {type(response.source_documents)}")
print(f"    数量: {len(response.source_documents)}")

print(f"\n[3] context (上下文):")
print(f"    类型: {type(response.context)}")
print(f"    长度: {len(response.context)} 字符")

print(f"\n[4] prompt (完整提示):")
print(f"    类型: {type(response.prompt)}")
print(f"    长度: {len(response.prompt)} 字符")

print("\n" + "=" * 50)
print("完整Prompt预览:")
print("-" * 50)
print(response.prompt[:500] + "..." if len(response.prompt) > 500 else response.prompt)

---

## 第四部分: 性能分析

### 4.1 分块效率分析

In [ ]:
# 分析不同分块大小的效率
long_doc = Document(content="机器学习是人工智能的核心。深度学习使用神经网络。自然语言处理理解语言。" * 20)

chunk_sizes = [50, 100, 200, 400]
overlap_ratios = [0.1, 0.2, 0.3]  # overlap占chunk_size的比例

efficiency_data = []

for size in chunk_sizes:
    for ratio in overlap_ratios:
        overlap = int(size * ratio)
        splitter_eff = RecursiveTextSplitter(
            chunk_size=size,
            chunk_overlap=overlap
        )
        chunks_eff = splitter_eff.split_text(long_doc.content)
        
        efficiency_data.append({
            'chunk_size': size,
            'overlap': overlap,
            'overlap_ratio': ratio,
            'num_chunks': len(chunks_eff),
            'total_chars': sum(len(c) for c in chunks_eff),
            'compression_ratio': len(long_doc.content) / sum(len(c) for c in chunks_eff)
        })

# 输出分析结果
print("\n分块效率分析：")
print(f"{'Size':<6} {'Overlap':<8} {'Ratio':<6} {'Chunks':<7} {'Total':<8} {'Compress':<8}")
print("-" * 60)
for d in efficiency_data:
    print(f"{d['chunk_size']:<6} {d['overlap']:<8} {d['overlap_ratio']:<6.1f} "
          f"{d['num_chunks']:<7} {d['total_chars']:<8} {d['compression_ratio']:.3f}")

### 4.2 检索质量评估

In [ ]:
# 测试检索相关性
test_cases = [
    {"query": "机器学习", "expected": ["机器学习", "人工智能"]},
    {"query": "神经网络", "expected": ["深度学习", "CNN", "RNN"]},
    {"query": "自然语言", "expected": ["NLP", "GPT", "BERT"]},
]

print("\n检索质量评估：")
print("=" * 60)

for case in test_cases:
    resp = pipeline.query(case["query"])
    source_contents = " ".join([doc.content for doc in resp.source_documents])
    
    # 计算命中关键词数
    hits = sum(1 for exp in case["expected"] if exp in source_contents)
    precision = hits / len(case["expected"]) if case["expected"] else 0
    
    print(f"\n查询: {case['query']}")
    print(f"期望关键词: {case['expected']}")
    print(f"命中数: {hits}/{len(case['expected'])}")
    print(f"精度: {precision:.2%}")

### 4.3 上下文长度优化

In [ ]:
# 测试不同max_context_length的影响
context_lengths = [200, 500, 1000, 2000, 5000]

test_query = "深度学习与机器学习的关系是什么？"

print("\n上下文长度优化测试：")
print("=" * 50)

for max_len in context_lengths:
    test_config = RAGConfig(
        chunk_size=200,
        top_k=3,
        max_context_length=max_len
    )
    test_pipeline = RAGPipeline(config=test_config)
    test_pipeline.add_documents(documents)
    
    test_resp = test_pipeline.query(test_query)
    actual_context_len = len(test_resp.context)
    
    print(f"\n最大上下文: {max_len}")
    print(f"实际上下文: {actual_context_len}")
    print(f"来源文档: {len(test_resp.source_documents)}")
    print(f"上下文占比: {actual_context_len/max_len:.1%}")

---

## 总结

本notebook介绍了RAG流水线的完整实现：

1. **文本分块**: RecursiveTextSplitter通过层级分隔符保持语义完整性
2. **RAG流水线**: 整合文档处理、检索和生成的端到端流程
3. **查询响应**: 支持灵活的查询方式和结构化响应
4. **性能优化**: 可通过调整chunk_size、overlap、top_k等参数优化效果